In [1]:
import numpy as np
from collections import Counter

In [2]:
def create_dataset():
    dataset = [
        [1, 1, 'yes'],
        [1, 1, 'yes'],
        [1, 0, 'no'],
        [0, 1, 'no'],
        [0, 1, 'no']
    ]
    feature_names = ['no surfacing','flippers']
    return dataset, feature_names

data, feature_names = create_dataset()
data, feature_names

([[1, 1, 'yes'], [1, 1, 'yes'], [1, 0, 'no'], [0, 1, 'no'], [0, 1, 'no']],
 ['no surfacing', 'flippers'])

In [3]:
# 3-1 Calculate Shannon entropy
def shannon_ent(dataset):
    num_entries = len(dataset)
    label_counts={}                           # The original method for label counts.
    for featvec in dataset:
        current_label = featvec[-1]
        if current_label not in label_counts:
            label_counts[current_label] = 1
        else:
            label_counts[current_label] += 1
    # label_ls= [featvec[-1] for featvec in dataset]  # The second method for label counts.
    # label_counts = Counter(label_ls)              
 
    shannon_ent = 0
    for key in label_counts:
        prob = label_counts[key] / num_entries
        shannon_ent += -prob * np.log2(prob) 
    return shannon_ent

shannon_ent(data)

np.float64(0.9709505944546686)

In [7]:
# 3-2 Split the dataset by designated feature.
def split_dataset(dataset, feat_index, value):
    ret_dataset = []
    for feat_vec in dataset:
        if feat_vec[feat_index] == value:
            reduced_feat_vec = feat_vec[:feat_index] + feat_vec[feat_index+1:]
            ret_dataset.append(reduced_feat_vec)
    return ret_dataset 

split_dataset(data, 0, 1)

[[1, 'yes'], [1, 'yes'], [0, 'no']]

In [12]:
# 3-3 Choose the best split
def best_feature_to_split(dataset):
    num_feat = len(dataset[0]) - 1
    base_ent = shannon_ent(dataset)

    best_info_gain = 0
    best_feat = -1
    
    for i in range(num_feat):
        feat_ls = [example[i] for example in dataset]
        unique_val = set(feat_ls)       
        new_ent = 0.0
        for value in unique_val:  
            sub_dataset = split_dataset(dataset, i, value)
            prob = len(sub_dataset) / len(dataset)     # No need to use float() in python 3.
            new_ent += prob * shannon_ent(sub_dataset)   

        info_gain = base_ent - new_ent     

        if info_gain > best_info_gain:       
            best_info_gain = info_gain        
            best_feat_idx = i
    return best_feat_idx                     


best_feature = best_feature_to_split(data)
best_feature

0

In [11]:
import operator

def majority_cnt(class_list):
    """
    Finds the most frequent class label in a list using a dictionary.
    
    :param class_list: List of class labels (e.g., ['yes', 'no', 'yes'])
    :return: The label that appears most frequently
    """
    # class_count = {}                  Method 1 for class counts.
    # for vote in class_list:
        # if vote not in class_count:
            # class_count[vote] = 0
        # class_count[vote] += 1
    
    classs_count = Counter(class_list)   # Method 2 for class counts.
    
    # Return the label with the highest count
    return classs_count.most_common(1)[0][0]


# Test section
if __name__ == "__main__":
    test_labels = [1, 1, 1, 0, 0, 2, 2, 2, 2]
    print(f"The majority class is: {majority_cnt(test_labels)}")

The majority class is: 2


In [13]:
# 3-4 Create trees

def create_tree(dataset, feature_names):
    """Recursively builds ID3 decision tree."""
    class_list = [example[-1] for example in dataset]

    # Stop if all samples have the same label.
    if class_list.count(class_list[0]) == len(class_list):
        return class_list[0]

    # Stop if no features left to split, only left the label column.
    if len(dataset[0]) == 1:
        return majority_cnt(class_list)

    # Select best feature and its name.
    best_feat_idx = best_feature_to_split(dataset)
    best_feat_name = feature_names[best_feat_idx]
    
    my_tree = {best_feat_name: {}}
    
    # Create new feature list excluding the current best feature
    remaining_names = feature_names[:best_feat_idx] + feature_names[best_feat_idx+1:]

    # Get unique values and split recursively
    unique_vals = set([example[best_feat_idx] for example in dataset])
    for value in unique_vals:
        sub_dataset = split_dataset(dataset, best_feat_idx, value)
        my_tree[best_feat_name][value] = create_tree(sub_dataset, remaining_names)

    return my_tree

# Usage
if __name__ == "__main__":
    data, labels = create_dataset()
    # Pass labels[:] to prevent original list modification
    tree = create_tree(data, labels[:])
    print(tree)

{'no surfacing': {0: 'no', 1: {'flippers': {0: 'no', 1: 'yes'}}}}


In [15]:
import pickle
import os


def store_tree(input_tree, file_path):
    """
    Serializes and saves the decision tree to a binary file.
    """
    # Create the directory if it doesn't exist
    os.makedirs(os.path.dirname(file_path), exist_ok=True)
    
    # Use 'wb' (write binary) for pickle in Python 3
    with open(file_path, 'wb') as fw:
        pickle.dump(input_tree, fw)
    
    print(f"Tree stored to {file_path} successfully.")


def get_tree(file_path):
    """
    Reads the serialized decision tree from a binary file.
    """
    # Use 'rb' (read binary) for pickle in Python 3
    with open(file_path, 'rb') as fr:
        return pickle.load(fr)


# Test the functions
if __name__ == "__main__":
    # Example tree structure
    my_tree = {'no surfacing': {0: 'no', 1: {'flippers': {0: 'no', 1: 'yes'}}}}
    file_path = "decisionTree/myTree.txt"
    
    # Store and then retrieve the tree
    store_tree(my_tree, file_path)
    retrieved_tree = get_tree(file_path)
    
    print("Retrieved tree structure:")
    print(retrieved_tree)

Tree stored to decisionTree/myTree.txt successfully.
Retrieved tree structure:
{'no surfacing': {0: 'no', 1: {'flippers': {0: 'no', 1: 'yes'}}}}
